In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import random
import numpy as np
import torch
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

In [9]:
import data
#data.download_pipeline() # 다운로드 필요시
df = data.preprocess_pipeline()

In [10]:
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 99149 entries, 0 to 99148
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   접수일자                 99149 non-null  datetime64[us]
 1   접수지역                 99149 non-null  category      
 2   배달지역                 99149 non-null  category      
 3   총중량_kg               99149 non-null  float64       
 4   연도                   99149 non-null  int32         
 5   월                    99149 non-null  int32         
 6   일                    99149 non-null  int32         
 7   요일                   99149 non-null  int32         
 8   공휴일여부                99149 non-null  int64         
 9   다음_공휴일까지_남은일수        99149 non-null  int64         
 10  지난_공휴일부터_지난일수        99149 non-null  int64         
 11  공휴일_직전_3일내           99149 non-null  int64         
 12  공휴일_직후_2일내           99149 non-null  int64         
 13  lag_1                99149 non-null  float

접수일자                   0
접수지역                   0
배달지역                   0
총중량_kg                 0
연도                     0
월                      0
일                      0
요일                     0
공휴일여부                  0
다음_공휴일까지_남은일수          0
지난_공휴일부터_지난일수          0
공휴일_직전_3일내             0
공휴일_직후_2일내             0
lag_1                  0
lag_2                  0
lag_7                  0
lag_14                 0
rolling_7_mean         0
rolling_7_std          0
rolling_14_mean        0
ratio_lag1_to_roll7    0
dtype: int64

In [11]:
df_train, df_test = data.split_timeseries_data(df)

📊 시계열 데이터셋 분할 완료
 전체 데이터 기간: 2025-01-20 ~ 2026-06-30
 Train 데이터   : 2025-01-20 ~ 2026-05-29 (93,077건)
 Test 데이터    : 2026-06-01 ~ 2026-06-30 (6,072건)


In [12]:
import lgbm, models
# 모델 학습 및 평가 실행
model_lgbm = lgbm.train_lgbm(df_train,df_test)
preds_lgbm = lgbm.predict_lgbm(model_lgbm, df_test)
models.evaluate_outputs(df_test["총중량_kg"], preds_lgbm)

LightGBM 모델 학습 시작...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008177 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2131
[LightGBM] [Info] Number of data points in the train set: 93077, number of used features: 14
[LightGBM] [Info] Start training from score 3601.162054


/home/oudedong/postoffice/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[200]	train's l1: 518.859	train's l2: 1.57106e+06	valid's l1: 535.913	valid's l2: 1.68153e+06

📈 모델 평가 결과
 MAE  (평균 절대 오차) : 536.50 kg
 RMSE (평균 제곱근 오차): 1,275.02 kg
 WAPE (가중 오차율)    : 15.15 %


In [13]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 6072 entries, 0 to 6071
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   접수일자                 6072 non-null   datetime64[us]
 1   접수지역                 6072 non-null   category      
 2   배달지역                 6072 non-null   category      
 3   총중량_kg               6072 non-null   float64       
 4   연도                   6072 non-null   int32         
 5   월                    6072 non-null   int32         
 6   일                    6072 non-null   int32         
 7   요일                   6072 non-null   int32         
 8   공휴일여부                6072 non-null   int64         
 9   다음_공휴일까지_남은일수        6072 non-null   int64         
 10  지난_공휴일부터_지난일수        6072 non-null   int64         
 11  공휴일_직전_3일내           6072 non-null   int64         
 12  공휴일_직후_2일내           6072 non-null   int64         
 13  lag_1                6072 non-null   float64

In [14]:
import lstm
# 1. 학습
model_lstm, label_encoders, scaler = lstm.train_lstm(df_train, epochs=30)

# 2. 예측 및 원본 인덱스 추출
preds_lstm, target_indices = lstm.predict_lstm(
    model_lstm, df_test, label_encoders, scaler
)

# 3. ⚠️ loc를 사용하여 원본 인덱스로 정답 슬라이싱 후 평가!
y_true_matched = df_test.loc[target_indices, "총중량_kg"].values

models.evaluate_outputs(y_true_matched, preds_lstm)

🖥️ 사용 장치: cuda

🚀 Categorical Embedding + LSTM 모델 학습 시작...
Epoch [01/30] | Loss (Huber): 0.04404
Epoch [05/30] | Loss (Huber): 0.02234
Epoch [10/30] | Loss (Huber): 0.01651
Epoch [15/30] | Loss (Huber): 0.01382
Epoch [20/30] | Loss (Huber): 0.01179
Epoch [25/30] | Loss (Huber): 0.00997
Epoch [30/30] | Loss (Huber): 0.00900

📈 모델 평가 결과
 MAE  (평균 절대 오차) : 424.07 kg
 RMSE (평균 제곱근 오차): 908.15 kg
 WAPE (가중 오차율)    : 10.81 %


In [18]:
w_lstm, w_lgb = 0.585, 0.415
final_pred = w_lstm * preds_lstm + w_lgb * preds_lgbm[target_indices]
models.evaluate_outputs(df_test.iloc[target_indices]["총중량_kg"],final_pred)


📈 모델 평가 결과
 MAE  (평균 절대 오차) : 384.75 kg
 RMSE (평균 제곱근 오차): 831.85 kg
 WAPE (가중 오차율)    : 9.81 %
